In [1]:
import os
import csv
import math
from pathlib import Path
from typing import List, Tuple, Iterable, Dict

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchmetrics.detection import MeanAveragePrecision

In [23]:
import torch

# Choose model type
MODEL_NAME = "dinov3_vitl16"

# Load model from GitHub (weights not needed, we'll load our checkpoint)
model = torch.hub.load(
    'facebookresearch/dinov3:main',  # GitHub repo
    MODEL_NAME,
    pretrained=False  # do NOT download weights
)

Using cache found in /home/rishabh.mondal/.cache/torch/hub/facebookresearch_dinov3_main


In [29]:
model

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-23): 24 x SelfAttentionBlock(
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=1024, out_features=3072, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    )
  )
  (norm)

In [ ]:
x = torch.randn(1, 3, 800, 800) 

In [31]:
model.get_intermediate_layers??

Signature:
model.get_intermediate_layers(
    x: torch.Tensor,
    *,
    n: Union[int, Sequence] = 1,
    reshape: bool = False,
    return_class_token: bool = False,
    return_extra_tokens: bool = False,
    norm: bool = True,
) -> Tuple[Union[torch.Tensor, Tuple[torch.Tensor, ...]]]
Docstring: <no docstring>
Source:   
    def get_intermediate_layers(
        self,
        x: torch.Tensor,
        *,
        n: Union[int, Sequence] = 1,  # Layers or n last layers to take
        reshape: bool = False,
        return_class_token: bool = False,
        return_extra_tokens: bool = False,
        norm: bool = True,
    ) -> Tuple[Union[torch.Tensor, Tuple[torch.Tensor, ...]]]:
        outputs = self._get_intermediate_layers_not_chunked(x, n)
        if norm:
            outputs_normed = []
            for out in outputs:
                if self.untie_cls_and_patch_norms:
                    x_norm_cls_reg = self.cls_norm(out[:, : self.n_storage_tokens + 1])
                    x_norm_p

In [ ]:
cls=model.get_intermediate_layers(x,return_class_token=True)
print(cls[-1][1].shape)                   

torch.Size([1, 1024])


In [1]:
import tarfile, os

tar_path = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/eyJsaW5rIjogInMzOi8vc3VwZXJ2aXNlbHktZGF0YXNldHMvMTk0MV94VmlldyAyMDE4L3h2aWV3LTIwMTgtRGF0YXNldE5pbmphLnRhciIsICJzaWciOiAiY2R0czhWeklYRXJVTTY3dExMUzlWZTVrT21UL2gyMzZuTHJhcHJldER5OD0ifQ==?response-content-disposition=attachment; filename=.1"
extract_path = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/extracted"

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(tar_path, "r") as tar:
    tar.extractall(path=extract_path)

print("Extraction complete:", extract_path)

FileNotFoundError: [Errno 2] No such file or directory: '/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/eyJsaW5rIjogInMzOi8vc3VwZXJ2aXNlbHktZGF0YXNldHMvMTk0MV94VmlldyAyMDE4L3h2aWV3LTIwMTgtRGF0YXNldE5pbmphLnRhciIsICJzaWciOiAiY2R0czhWeklYRXJVTTY3dExMUzlWZTVrT21UL2gyMzZuTHJhcHJldER5OD0ifQ==?response-content-disposition=attachment; filename=.1'